In [ ]:
!uv pip install pandas
!uv pip install numpy
!uv pip install matplotlib
!uv pip install seaborn
!uv pip install scikit-learn
!uv pip install tensorflow
!uv pip install keras-tuner

!uv pip install ruff

In [ ]:
# Import core data manipulation and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from pathlib import Path

In [ ]:
# Sets plotting style for professional, report-ready graphics
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "serif" # Matches Times New Roman aesthetic

In [ ]:
dataset_path = Path("dataset/rees46_customer_model.csv")
if not dataset_path.exists():
    raise FileNotFoundError(
        f"{dataset_path} is missing. Download/bootstrap the dataset first."
    )

df = pd.read_csv(dataset_path)

In [ ]:
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns")

display(df.head(10))

print("--- Data Types & Missing Values ---")
df.info()

In [ ]:
list(df.columns)

In [ ]:
list(df.nunique())

In [ ]:
# Identify the target column 
target_col = "target_event" # Usually 0 for churned, 1 for retained

# Calculate the distribution of the target variable
churn_counts = df[target_col].value_counts()
churn_percentages = df[target_col].value_counts(normalize=True) * 100

print("Target Event Distribution (0 = Churn/No Action, 1 = Retained/Action):\n", churn_counts)
print("\nTarget Event Percentages:\n", churn_percentages)

In [ ]:
# Plotting the class distribution for the report
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x=target_col, hue=target_col, palette="viridis", legend=False)
plt.title("Distribution of Target Event (Churn vs. Retained)", fontsize=14, fontweight="bold")
plt.xlabel("Target Event (0 = Churn, 1 = Retained)", fontsize=12)
plt.ylabel("Number of Customers", fontsize=12)

# Annotate bars with exact percentages to show class imbalance clearly
for p in ax.patches:
    percentage = f"{100 * p.get_height() / len(df):.1f}%"
    x = p.get_x() + p.get_width() / 2 - 0.1
    y = p.get_height() + 1000
    ax.annotate(percentage, (x, y), size=12)

plt.tight_layout()
plt.show()

In [ ]:
# --- DATA CLEANING & PREVENTION OF TARGET LEAKAGE ---

# 1. Remove identifier columns as they hold no predictive mathematical value
cols_to_drop = ["row_id", "user_id", "time_step"]

# 2. Remove other 'target_' columns to prevent data leakage. 
# If the model knows the future 'target_revenue', it will cheat to guess 'target_event'.
leakage_cols =["target_revenue", "target_customer_value", "target_customer_value_lag1", "target_actual_profit"]

# Combine the lists and drop them from the dataframe
df_clean = df.drop(columns=cols_to_drop + leakage_cols)

print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape} (Removed IDs and Data Leaks)")

In [ ]:
# Separate the dataset into Features (X) and Target (y)
X = df_clean.drop(columns=[target_col])
y = df_clean[target_col]

# Split the data: 70% for Training, 15% for Validation, 15% for Testing
# Stratify=y ensures the 0s and 1s are balanced across all splits
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

In [ ]:
# Standardizes features by removing the mean and scaling to unit variance (z-score)
scaler = StandardScaler()

# Fit the scaler only on the training data to prevent information leakage from the test set
X_train_scaled = scaler.fit_transform(X_train)

# Apply the trained scaler to the validation and test sets
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training Data Shape:   {X_train_scaled.shape}")
print(f"Validation Data Shape: {X_val_scaled.shape}")
print(f"Testing Data Shape:    {X_test_scaled.shape}")